In [4]:
import json
import matplotlib.pyplot as plt
import pandas as pd
import requests

from pathlib import Path
from requests.exceptions import HTTPError, RequestException


In [5]:
API_URL = "https://jsonplaceholder.typicode.com/users"

In [7]:
def get_customers(url: str = API_URL) -> list[dict]:
    """Fetch the customer list. Report failures clearly instead of crashing."""
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        data = response.json()
        print(f"Successfully fetched data from {url}. Number of records: {len(data)}")
        return data
    except HTTPError as http_err:
        print(f"HTTP error occurred while fetching data from {url}: {http_err}")
        return []
    except requests.RequestException as e:
        print(f"Error fetching data from {url}: {e}")
        return []   

In [ ]:
get_customers()  # Fetch and print the customer list

In [8]:
def explore(record: dict, indent: int = 0) -> None:
    """Print every key in a (possibly nested) record with its type, indented by depth."""
    for key, value in record.items():
        print(" " * indent + f"{key}: {type(value).__name__}")
        if isinstance(value, dict):
            explore(value, indent + 2)

In [10]:
explore(get_customers()[0])  # Fetch and print the customer list

Successfully fetched data from https://jsonplaceholder.typicode.com/users. Number of records: 10
id: int
name: str
username: str
email: str
address: dict
  street: str
  suite: str
  city: str
  zipcode: str
  geo: dict
    lat: str
    lng: str
phone: str
website: str
company: dict
  name: str
  catchPhrase: str
  bs: str


In [ ]:
def extract_attributes(customers: list[dict]) -> list[dict]:
    """Return one flat dict per customer with the attributes listed in the task."""
    '''Implementing extract_attributes(customers) to return one flat dict per customer with id, name, username, email, email domain, phone, city, zipcode, latitude, longitude, company and website. Latitude and longitude must be floats.'''
    extracted = []
    for customer in customers:
        flat_customer = {
            "id": customer.get("id"),
            "name": customer.get("name"),
            "username": customer.get("username"),
            "email": customer.get("email"),
            "email_domain": customer.get("email").split("@")[-1] if customer.get("email") else None,
            "phone": customer.get("phone"),
            "city": customer.get("address", {}).get("city"),
            "zipcode": customer.get("address", {}).get("zipcode"),
            "latitude": float(customer.get("address", {}).get("geo", {}).get("lat")) if customer.get("address", {}).get("geo", {}).get("lat") else None,
            "longitude": float(customer.get("address", {}).get("geo", {}).get("lng")) if customer.get("address", {}).get("geo", {}).get("lng") else None,
            "company": customer.get("company", {}).get("name"),
            "website": customer.get("website")
        }
        extracted.append(flat_customer)
    return extracted

In [16]:
extracted_customers = extract_attributes(get_customers())
print(json.dumps(extracted_customers, indent=2))  # Print the extracted customer attributes

Successfully fetched data from https://jsonplaceholder.typicode.com/users. Number of records: 10
[
  {
    "id": 1,
    "name": "Leanne Graham",
    "username": "Bret",
    "email": "Sincere@april.biz",
    "email_domain": "april.biz",
    "phone": "1-770-736-8031 x56442",
    "city": "Gwenborough",
    "zipcode": "92998-3874",
    "latitude": -37.3159,
    "longitude": 81.1496,
    "company": "Romaguera-Crona",
    "website": "hildegard.org"
  },
  {
    "id": 2,
    "name": "Ervin Howell",
    "username": "Antonette",
    "email": "Shanna@melissa.tv",
    "email_domain": "melissa.tv",
    "phone": "010-692-6593 x09125",
    "city": "Wisokyburgh",
    "zipcode": "90566-7771",
    "latitude": -43.9509,
    "longitude": -34.4618,
    "company": "Deckow-Crist",
    "website": "anastasia.net"
  },
  {
    "id": 3,
    "name": "Clementine Bauch",
    "username": "Samantha",
    "email": "Nathan@yesenia.net",
    "email_domain": "yesenia.net",
    "phone": "1-463-123-4447",
    "city": "McK

In [17]:
def unique_cities(customers: list[dict]) -> set[str]:
    """Return the set of distinct cities."""
    return {customer.get("city") for customer in customers if customer.get("city")}

In [20]:
unique_cities_set = unique_cities(extracted_customers)
print(f"We have {len(extracted_customers)} customers distributed across {len(unique_cities_set)} unique cities: {unique_cities_set}")  # Print the set of unique cities

We have 10 customers distributed across 10 unique cities: {'McKenziehaven', 'Howemouth', 'Bartholomebury', 'Aliyaview', 'Lebsackbury', 'Wisokyburgh', 'Roscoeview', 'South Christy', 'Gwenborough', 'South Elvis'}


In [ ]:
def city_counts(customers: list[dict]) -> dict[str, int]:
    """Count customers per city using a loop and/or a dict comprehension (no Counter, no pandas)."""
    counts = {}
    for customer in customers:
        city = customer.get("city")
        if city:
            counts[city] = counts.get(city, 0) + 1
    return counts


In [24]:
city_count_dict = city_counts(extracted_customers)
print("Customer counts per city (sorted from most to least common):")
for city, count in sorted(city_count_dict.items(), key=lambda item: item[1], reverse=True):
    print(f"{city}: {count}")

Customer counts per city (sorted from most to least common):
Gwenborough: 1
Wisokyburgh: 1
McKenziehaven: 1
South Elvis: 1
Roscoeview: 1
South Christy: 1
Howemouth: 1
Aliyaview: 1
Bartholomebury: 1
Lebsackbury: 1


In [26]:
def email_domain_counts(customers: list[dict]) -> dict[str, int]:
    """Count customers per email top-level domain (e.g. 'biz', 'tv')."""
    '''Implement email_domain_counts(customers) using a dict comprehension. Print both sorted from most to least common.'''
    counts = {}
    for customer in customers:
        email = customer.get("email")
        if email and "@" in email:
            domain = email.split("@")[-1].split(".")[-1]
            counts[domain] = counts.get(domain, 0) + 1
    return counts

In [27]:
email_domain_count_dict = email_domain_counts(extracted_customers)
print("Customer counts per email top-level domain (sorted from most to least common):") 
for domain, count in sorted(email_domain_count_dict.items(), key=lambda item: item[1], reverse=True):
    print(f"{domain}: {count}")

Customer counts per email top-level domain (sorted from most to least common):
biz: 3
tv: 1
net: 1
org: 1
ca: 1
info: 1
me: 1
io: 1


In [28]:
def to_dataframe(rows: list[dict]) -> pd.DataFrame:
    """Convert a list of dicts to a pandas DataFrame."""
    return pd.DataFrame(rows)

In [29]:
to_dataframe(extracted_customers)  # Convert the extracted customer attributes to a DataFrame

,id,name,username,email,email_domain,phone,city,zipcode,latitude,longitude,company,website
0,1,Leanne Graham,Bret,Sincere@april.biz,april.biz,1-770-736-8031 x56442,Gwenborough,92998-3874,-37.3159,81.1496,Romaguera-Crona,hildegard.org
1,2,Ervin Howell,Antonette,Shanna@melissa.tv,melissa.tv,010-692-6593 x09125,Wisokyburgh,90566-7771,-43.9509,-34.4618,Deckow-Crist,anastasia.net
2,3,Clementine Bauch,Samantha,Nathan@yesenia.net,yesenia.net,1-463-123-4447,McKenziehaven,59590-4157,-68.6102,-47.0653,Romaguera-Jacobson,ramiro.info
3,4,Patricia Lebsack,Karianne,Julianne.OConner@kory.org,kory.org,493-170-9623 x156,South Elvis,53919-4257,29.4572,-164.2990,Robel-Corkery,kale.biz
4,5,Chelsey Dietrich,Kamren,Lucio_Hettinger@annie.ca,annie.ca,(254)954-1289,Roscoeview,33263,-31.8129,62.5342,Keebler LLC,demarco.info
5,6,Mrs. Dennis Schulist,Leopoldo_Corkery,Karley_Dach@jasper.info,jasper.info,1-477-935-8478 x6430,South Christy,23505-1337,-71.4197,71.7478,Considine-Lockman,ola.org
6,7,Kurtis Weissnat,Elwyn.Skiles,Telly.Hoeger@billy.biz,billy.biz,210.067.6132,Howemouth,58804-1099,24.8918,21.8984,Johns Group,elvis.io
7,8,Nicholas Runolfsdottir V,Maxime_Nienow,Sherwood@rosamond.me,rosamond.me,586.493.6943 x140,Aliyaview,45169,-14.3990,-120.7677,Abernathy Group,jacynthe.com
8,9,Glenna Reichert,Delphine,Chaim_McDermott@dana.io,dana.io,(775)976-6794 x41206,Bartholomebury,76495-3109,24.6463,-168.8889,Yost and Sons,conrad.com
9,10,Clementina DuBuque,Moriah.Stanton,Rey.Padberg@karina.biz,karina.biz,024-648-3804,Lebsackbury,31428-2261,-38.2386,57.2232,Hoeger LLC,ambrose.net


In [35]:
#Counting Cities using values_counts
df = to_dataframe(extracted_customers)
city_counts = df['city'].value_counts()
print("City counts:")
print(city_counts)

City counts:
city
Gwenborough       1
Wisokyburgh       1
McKenziehaven     1
South Elvis       1
Roscoeview        1
South Christy     1
Howemouth         1
Aliyaview         1
Bartholomebury    1
Lebsackbury       1
Name: count, dtype: int64


In [37]:
#Printing the shape of the DataFrame
df = to_dataframe(extracted_customers)
print(f"Shape of the DataFrame: {df.shape}")

Shape of the DataFrame: (10, 12)


In [34]:
#Printing the dtypes of the DataFrame
print("Data types of the DataFrame columns:")
print(df.dtypes)

Data types of the DataFrame columns:
id                int64
name             object
username         object
email            object
email_domain     object
phone            object
city             object
zipcode          object
latitude        float64
longitude       float64
company          object
website          object
dtype: object


In [39]:
#Exporting the customers to a CSV file
'''Required columns: customer_id,name,username,email,email_domain,phone,city,zipcode,latitude,longitude,company,website'''
df.to_csv("output/customers.csv", index=False)

In [40]:
# Exporting Customercounts to a CSV file
'''Required columns: city,customer_count'''
city_counts.to_csv("output/city_counts.csv", header=['count'])

In [41]:
def plot_distribution(counts: dict[str, int], path: Path) -> None:
    """Plot a bar chart of the distribution of counts and save it to the specified path."""
    plt.figure(figsize=(10, 6))
    plt.bar(counts.keys(), counts.values())
    plt.xlabel('Category')
    plt.ylabel('Count')
    plt.title('Distribution of Counts')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(path)
    plt.close()

In [31]:
plot_distribution(city_count_dict, Path("city_distribution.png"))

In [42]:
if __name__ == "__main__":
    print("Starting the data analysis script...")
    print("Fetching customer data...")
    print("Analyzing customer data...")
    print("Generating reports...")
    pass

Starting the data analysis script...
Fetching customer data...
Analyzing customer data...
Generating reports...
